In [1]:
import os

In [2]:
%pwd

'c:\\Projects\\Kidney-Disease-Classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Projects\\Kidney-Disease-Classification'

In [6]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [8]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [10]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [12]:
import os
import zipfile
import gdown
from cnnClassifier.utils.common import get_size
from cnnClassifier import logger

In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        '''
        Download file from Google Drive using gdown library.
        '''

        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading file from {dataset_url} to {zip_download_dir}")

            file_id = dataset_url.split("/")[-2]
            prefix = "https://drive.google.com/uc?/export=download&id="
            gdown.download(prefix + file_id, zip_download_dir, quiet=False)

            logger.info(f"File downloaded from {dataset_url} to {zip_download_dir}")
        except Exception as e:
            raise e

    def extract_zip_file(self):
        """
        Extract the downloaded zip file to the specified directory.
        """

        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [15]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-07-17 18:06:36,782: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-17 18:06:36,788: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-17 18:06:36,790: INFO: common: created directory at: artifacts]
[2026-07-17 18:06:36,792: INFO: common: created directory at: artifacts/data_ingestion]
[2026-07-17 18:06:36,794: INFO: 1519392221: Downloading file from https://drive.google.com/file/d/1CksUeT9k3NUMWXWR1AWDGSv_-T9bcNqX/view?usp=sharing to artifacts/data_ingestion/dataset.zip]


Downloading...
From (original): https://drive.google.com/uc?id=1CksUeT9k3NUMWXWR1AWDGSv_-T9bcNqX
From (redirected): https://drive.google.com/uc?id=1CksUeT9k3NUMWXWR1AWDGSv_-T9bcNqX&confirm=t&uuid=b5ef1cbc-3c67-410a-a75e-91e36bab1ab2
To: c:\Projects\Kidney-Disease-Classification\artifacts\data_ingestion\dataset.zip
100%|██████████| 1.63G/1.63G [07:50<00:00, 3.46MB/s]


EnsureError: Argument path of type <class 'str'> to <function get_size at 0x000001B7239089D0> does not match annotation type <class 'pathlib.Path'>